In [1]:
import numpy as np
import pandas as pd
import openmdao.api as om

from standard_evaluator import StandardBase, StandardGroup, OptionsDictionaryUnit

In [2]:
class Paraboloid(StandardBase, om.ExplicitComponent):
    """
    Evaluates the equation f(x,y) = (|x|-3)^2 + |x|y + (y+4)^2 - 3.
    """

    @classmethod
    def _define_options(cls) -> OptionsDictionaryUnit:
        """Abstract method that allows a developer to define information about the parameters this class uses.

        This method is called by a class method that is adding options to the `class_options` option.

        Returns:
            OptionsDictionaryUnit -- The required options for this class, their default values, and, if required, their units.
        """
        options = OptionsDictionaryUnit()
        options.declare("dimensions", default=40, types=int)
        return options

    def setup(self):
        
        self.add_standard_input('x', val=np.ones(self.lookup_option_value("dimensions")), options=["dimensions"],
            look_up=False,)
        self.add_input('y', val=0.0)

        self.add_output('f_xy', val=0.0)

    def setup_partials(self):
        # Finite difference all partials.
        self.declare_partials('*', '*', method='fd')

    def compute(self, inputs, outputs):
        """
        f(x,y) = (x-3)^2 + xy + (y+4)^2 - 3

        Minimum at: x = 6.6667; y = -7.3333
        """
        x = inputs['x']
        y = inputs['y']
        outputs['f_xy'] = (np.linalg.norm(x) - 3.0)**2 + np.linalg.norm(x) * y + (y + 4.0)**2 - 3.0

class MyGroup(StandardGroup, om.Group):
    """
    Prepare derived values of aircraft geometry for aerodynamics analysis.
    """

    def setup(self):
        class_options = self.options["class_options"]


        self.add_subsystem(
            "parab_comp",
            Paraboloid(class_options=class_options),
            promotes_inputs=["*"],
        )



if __name__ == "__main__":

    print(Paraboloid.required_options())
    model = MyGroup()
    print(model.required_options())

    prob = om.Problem(model)
    prob.setup()

    my_options = model.full_options()
    my_options.set(dimensions=100)

    model = MyGroup(class_options=my_options)
    prob = om.Problem(model)
    prob.setup()
    print(model.full_options())
    print(model.current_options())


    prob.set_val('parab_comp.x', 3.0)
    prob.set_val('parab_comp.y', -4.0)

    prob.run_model()
    print(prob.get_val('parab_comp.f_xy'))
    print(prob.get_val('parab_comp.x'))


==========  =======  =================  ================  ===========  ========
Option      Default  Acceptable Values  Acceptable Types  Description  Units   
==========  =======  =================  ================  ===========  ========
dimensions       40  N/A                ['int']                        unitless
==========  =======  =================  ================  ===========  ========
======  =======  =================  ================  ===========  =====
Option  Default  Acceptable Values  Acceptable Types  Description  Units
======  =======  =================  ================  ===========  =====
                                                                        
======  =======  =================  ================  ===========  =====
Looking up value for dimensions from the full options method
{'dimensions'}
{'dimensions'}
==========  =======  =================  ================  ===========  ========
Option      Default  Acceptable Values  Acceptable Types  Descri